# Extreme cases - results

Analysis of the five extreme-case runs against the reference run. The scenario folders sit in the Balmorel root (built by `extreme_cases/make_extreme_cases.py`, see `extreme_cases/README.md`). All five simulate the year 2035, weeks S02 and S28, and Thursday to Saturday every 3 hours.

| Scenario | Change | Compared in this notebook |
|---|---|---|
| `ext_reference` | none | reference for all comparisons |
| `ext_no_pv` | no PV anywhere (solar heat allowed) | new and installed PV capacity, objective |
| `ext_onshore_wind_capex_x10` | investment cost of new onshore wind x 10 | installed capacity, objective |
| `ext_no_transmission_expansion` | no new transmission capacity | installed capacity, objective |
| `ext_no_france_interconnectors` | all links of France removed, no new ones | map of the directed transmission lines, objective |

**How to use.** Download the results of each run with `get_results.ps1` (they land in `<scenario>/model/results/MainResults.gdx`) and run all cells. A section whose results are missing prints a message and is skipped, so the notebook can be run while only some runs are finished. `ext_reference` is needed for every comparison.

Units: capacity in GW, objective in Mmoney (the objective is the sum of the cost categories of `OBJ_YCR` over all regions for the simulated year).

In [ ]:
import io
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- settings ---------------------------------------------------------------
ROOT = Path(r"C:\Balmorel")
RESULTS_ROOT = ROOT            # folder that holds the scenario folders: <scenario>/model/results/MainResults.gdx
GDXDUMP = Path(shutil.which("gdxdump") or r"C:\GAMS\50\gdxdump.exe")      # reads the .gdx result files
YEAR = 2035                    # the simulated year
USE_CARTOPY_BACKGROUND = True  # coastlines and borders on the maps if cartopy is installed (needs internet the first time)

REFERENCE = "ext_reference"
CASES = {"ext_no_pv": "No PV", "ext_onshore_wind_capex_x10": "Onshore wind capex x10",
         "ext_no_transmission_expansion": "No transmission expansion", "ext_no_france_interconnectors": "No France interconnectors"}
# one colour per scenario in every figure (colour follows the scenario, never its position)
REF_COLOR = "#7a7a7a"
CASE_COLOR = {"ext_no_pv": "#eda100", "ext_onshore_wind_capex_x10": "#1baf7a",
              "ext_no_transmission_expansion": "#4a3aa7", "ext_no_france_interconnectors": "#e87ba4"}
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "0.88", "grid.linewidth": 0.6, "font.size": 9})

_cache = {}


def result_file(scenario):
    return RESULTS_ROOT / scenario / "model" / "results" / "MainResults.gdx"


def symbol(scenario, name):
    "One result table of a scenario as a DataFrame (read once with gdxdump)."
    if (scenario, name) not in _cache:
        out = subprocess.run([str(GDXDUMP), str(result_file(scenario)), f"symb={name}", "format=csv"],
                             capture_output=True, text=True, check=True).stdout
        _cache[scenario, name] = pd.read_csv(io.StringIO(out))
    return _cache[scenario, name]


def ready(case=None):
    "True when the reference (and the case) have results; otherwise says what is missing."
    missing = [s for s in ([REFERENCE] + ([case] if case else [])) if not result_file(s).exists()]
    if missing:
        print("Results not available yet for:", ", ".join(missing), "- section skipped.")
    return not missing


status = pd.DataFrame({"results": {s: "available" if result_file(s).exists() else "missing" for s in [REFERENCE, *CASES]}})
display(status)

## 1. Objective function

The cost categories of `OBJ_YCR` are summed over all regions and grouped into seven groups. The same function is used for the per-scenario figures below.

In [ ]:
GROUPS = ["Generation capital", "Generation fixed O&M", "Generation variable O&M", "Fuel", "CO2 tax and transport",
          "Transmission capital", "Other (H2, heat network, transmission O&M)"]
GROUP_COLOR = dict(zip(GROUPS, PAL))
GROUP_OF = {"GENERATION_CAPITAL_COSTS": GROUPS[0], "GENERATION_FIXED_COSTS": GROUPS[1], "GENERATION_OPERATIONAL_COSTS": GROUPS[2],
            "GENERATION_FUEL_COSTS": GROUPS[3], "GENERATION_CO2_TAX": GROUPS[4], "GENERATION_CO2_TRANSPORT": GROUPS[4],
            "TRANSMISSION_CAPITAL_COSTS": GROUPS[5]}


def objective_by_group(scenario):
    "Objective (Mmoney) of the simulated year per cost group."
    obj = symbol(scenario, "OBJ_YCR")
    obj = obj[obj["Y"] == YEAR]
    return obj.groupby(obj["SUBCATEGORY"].map(GROUP_OF).fillna(GROUPS[-1]))["Val"].sum().reindex(GROUPS).fillna(0.0)


def plot_objective(case):
    "Reference and scenario objective stacked by cost group, and the change per group."
    ref, scn = objective_by_group(REFERENCE), objective_by_group(case)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.6]})
    for i, (label, values) in enumerate([("reference", ref), (CASES[case], scn)]):
        bottom = 0.0
        for g in GROUPS:
            ax1.bar(i, values[g], bottom=bottom, color=GROUP_COLOR[g], width=0.6, edgecolor="white", linewidth=1.2,
                    label=g if i == 0 else None)
            bottom += values[g]
        ax1.text(i, bottom, f"{bottom:,.0f}", ha="center", va="bottom", fontsize=8.5)
    ax1.set_xticks([0, 1], ["reference", CASES[case]], fontsize=8.5)
    ax1.set_ylabel("Mmoney")
    ax1.set_title("Objective function", loc="left", fontsize=10)
    ax1.grid(axis="x", visible=False)
    diff = scn - ref
    ax2.barh(range(len(GROUPS)), diff.values, color=CASE_COLOR[case], height=0.6)
    ax2.set_yticks(range(len(GROUPS)), GROUPS, fontsize=8)
    ax2.invert_yaxis()
    ax2.axvline(0, color="0.3", lw=0.8)
    ax2.set_xlabel("change vs reference (Mmoney)")
    total = scn.sum() - ref.sum()
    ax2.set_title(f"Change per cost group; total {total:+,.0f} Mmoney ({total / ref.sum():+.1%})", loc="left", fontsize=10)
    ax1.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    fig.tight_layout()
    plt.show()


if ready():
    rows = []
    for case in CASES:
        if result_file(case).exists():
            ref_total, case_total = objective_by_group(REFERENCE).sum(), objective_by_group(case).sum()
            rows.append({"scenario": case, "reference": ref_total, "scenario value": case_total,
                         "change": case_total - ref_total, "change %": (case_total - ref_total) / ref_total * 100})
    if rows:
        summary = pd.DataFrame(rows).set_index("scenario")
        display(summary.style.format({"reference": "{:,.0f}", "scenario value": "{:,.0f}", "change": "{:+,.0f}", "change %": "{:+.2f}"}))
        fig, ax = plt.subplots(figsize=(8, 0.6 * len(summary) + 1.2))
        ax.barh(range(len(summary)), summary["change %"], color=[CASE_COLOR[c] for c in summary.index], height=0.6)
        ax.set_yticks(range(len(summary)), [CASES[c] for c in summary.index])
        ax.invert_yaxis()
        ax.axvline(0, color="0.3", lw=0.8)
        ax.set_xlabel("objective change vs reference (%)")
        ax.grid(axis="y", visible=False)
        plt.tight_layout()
        plt.show()

## 2. Capacity helpers

`G_CAP_YCRAF` holds the generation capacity of the simulated year in three categories: exogenous (existing), endogenous (new investment) and decommissioning (negative). **New capacity** = endogenous. **Installed capacity** = all three added up. By default the comparison uses electricity-producing technologies (storage with electricity as commodity included).

In [ ]:
def capacity(scenario, kind="installed", commodity="ELECTRICITY", by="TECH_TYPE", tech=None):
    "Capacity (GW) of the simulated year, grouped by `by` (TECH_TYPE, C = country or RRR = region)."
    g = symbol(scenario, "G_CAP_YCRAF")
    g = g[(g["Y"] == YEAR) & (g["COMMODITY"] == commodity)]
    if tech is not None:
        g = g[g["TECH_TYPE"] == tech]
    if kind == "new":
        g = g[g["VARIABLE_CATEGORY"] == "ENDOGENOUS"]
    return g.groupby(by)["Val"].sum()


def compare_bars(case, ref, scn, title, xlabel="GW", min_value=0.05, sort_by="reference", panel_widths=(2, 1.3)):
    "Horizontal bars reference vs scenario, and the difference, for the categories of two Series."
    df = pd.concat({"reference": ref, CASES[case]: scn}, axis=1).fillna(0.0)
    df = df[df.abs().max(axis=1) >= min_value].sort_values(sort_by)
    if df.empty:
        print("nothing to show:", title)
        return df
    y = np.arange(len(df))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 0.45 * len(df) + 2), sharey=True, gridspec_kw={"width_ratios": list(panel_widths)})
    ax1.barh(y - 0.2, df["reference"], height=0.38, color=REF_COLOR, label="reference")
    ax1.barh(y + 0.2, df[CASES[case]], height=0.38, color=CASE_COLOR[case], label=CASES[case])
    ax1.set_yticks(y, df.index, fontsize=8)
    ax1.set_xlabel(xlabel)
    ax1.set_title(title, loc="left", fontsize=10)
    ax1.legend(frameon=False, fontsize=8, loc="lower right")
    ax1.grid(axis="y", visible=False)
    diff = df[CASES[case]] - df["reference"]
    ax2.barh(y, diff, height=0.6, color=CASE_COLOR[case])
    ax2.axvline(0, color="0.3", lw=0.8)
    ax2.set_xlabel(f"change vs reference ({xlabel})")
    ax2.set_title("Change", loc="left", fontsize=10)
    ax2.grid(axis="y", visible=False)
    fig.tight_layout()
    plt.show()
    return df.assign(change=diff)[["reference", CASES[case], "change"]]


def installed_capacity_figure(case, commodity="ELECTRICITY"):
    "Installed capacity by technology type, reference vs scenario."
    return compare_bars(case, capacity(REFERENCE, "installed", commodity), capacity(case, "installed", commodity),
                        f"Installed {commodity.lower()} capacity {YEAR} by technology")

### Transmission map helpers

Directed transmission capacity of the simulated year (existing plus new) as arrows between approximate region centres; used for the PV and France sections.

In [ ]:
# approximate centre of every region: (longitude, latitude)
REGION_XY = {
    "AL": (20.0, 41.1), "AT": (14.5, 47.5), "BA": (17.8, 44.2), "BE": (4.7, 50.6), "BG": (25.3, 42.7), "CH": (8.2, 46.8),
    "CY": (33.2, 35.0), "CZ": (15.5, 49.8), "DE4-E": (12.8, 51.8), "DE4-N": (9.8, 53.3), "DE4-S": (10.5, 48.9), "DE4-W": (7.5, 51.4),
    "DK1": (9.3, 56.0), "DK2": (11.9, 55.4), "EE": (25.5, 58.7), "ES": (-3.7, 40.0), "FIN": (26.0, 62.5), "FR": (2.5, 46.6),
    "GR": (22.0, 39.5), "HR": (15.5, 45.2), "HU": (19.3, 47.2), "IE": (-8.0, 53.2), "IT": (12.5, 42.8), "LT": (23.9, 55.3),
    "LU": (6.1, 49.8), "LV": (24.8, 56.9), "ME": (19.2, 42.7), "MK": (21.7, 41.6), "MT": (14.4, 35.9), "NL": (5.5, 52.2),
    "NO1": (9.5, 60.0), "NO2": (7.0, 59.0), "NO3": (10.0, 63.5), "NO4": (17.0, 68.5), "NO5": (6.5, 60.8), "PL": (19.4, 52.0),
    "PT": (-8.0, 39.5), "RO": (24.9, 45.9), "RS": (20.8, 44.0), "SE1": (20.0, 66.5), "SE2": (16.5, 63.0), "SE3": (15.0, 59.5),
    "SE4": (14.0, 56.2), "SI": (14.8, 46.1), "SK": (19.5, 48.7), "TR": (35.0, 39.0), "UK": (-1.5, 53.0),
}
FRANCE_COLOR, LINE_COLOR, MORE_COLOR = "#d6336c", "#2a78d6", "#1baf7a"


def links(scenario):
    "Directed transmission capacity (GW) of the simulated year, summed over existing and new capacity."
    x = symbol(scenario, "X_CAP_YCR")
    x = x[x["Y"] == YEAR]
    return x.groupby(["IRRRE", "IRRRI"])["Val"].sum()


def draw_links(ax, values, title, transform=None, color=None, width_per_gw=0.35, min_gw=0.05, max_width=4.5, label_nodes=True):
    "Arrows between region centres: one per directed link, width proportional to |value|."
    xycoords = transform if transform is not None else "data"
    if not (values.abs() >= min_gw).any():
        ax.text(0.5, 0.5, "no line changes", transform=ax.transAxes, ha="center", color="0.4", fontsize=11)
    for (a, b), v in values.items():
        if abs(v) < min_gw or a not in REGION_XY or b not in REGION_XY:
            continue
        c = color(a, b, v) if color else (FRANCE_COLOR if "FR" in (a, b) else LINE_COLOR)
        ax.annotate("", xy=REGION_XY[b], xytext=REGION_XY[a], xycoords=xycoords, textcoords=xycoords,
                    arrowprops=dict(arrowstyle="-|>", lw=min(max_width, 0.3 + width_per_gw * abs(v)), color=c, alpha=0.75,
                                    shrinkA=3, shrinkB=3, mutation_scale=6, connectionstyle="arc3,rad=0.12"))
    lons, lats = zip(*REGION_XY.values())
    kw = {"transform": transform} if transform is not None else {}
    ax.scatter(lons, lats, s=10, color="0.25", zorder=5, **kw)
    if label_nodes:
        for r, (lon, lat) in REGION_XY.items():
            ax.text(lon, lat + 0.5, r, fontsize=5.5, ha="center", color="0.25", zorder=6, **kw)
    ax.set_title(title, loc="left", fontsize=10)


def map_figure(panels, background):
    "panels: [(title, values, colour function or None)]; background: draw coastlines and borders with cartopy."
    if background:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        fig = plt.figure(figsize=(6.5 * len(panels), 7.5))
        axes = [fig.add_subplot(1, len(panels), i + 1, projection=ccrs.PlateCarree()) for i in range(len(panels))]
    else:
        fig, axes = plt.subplots(1, len(panels), figsize=(6.5 * len(panels), 7.5))
    for ax, (title, values, color) in zip(axes, panels):
        transform = None
        if background:
            ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.5, color="0.55")
            ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3, color="0.75")
            transform = ccrs.PlateCarree()._as_mpl_transform(ax)
        draw_links(ax, values, title, transform=transform, color=color)
        ax.set_xlim(-12, 37)
        ax.set_ylim(33, 71)
        if not background:
            ax.set_aspect(1 / np.cos(np.radians(52)))
            ax.grid(False)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.tight_layout()
    return fig


def show_maps(panels):
    "Draw with a cartopy background if wanted and possible, otherwise without."
    if USE_CARTOPY_BACKGROUND:
        try:
            fig = map_figure(panels, background=True)
            fig.canvas.draw()                 # a missing coastline file or no internet shows up here
            plt.show()
            return
        except Exception as error:            # cartopy not installed, or its map data cannot be downloaded
            plt.close("all")
            print(f"Map background not available ({type(error).__name__}); drawing without coastlines.")
    map_figure(panels, background=False)
    plt.show()

## 3. No PV (`ext_no_pv`)

Expected: no PV generation and no new PV capacity, more wind and other capacity, higher objective. Installed PV = existing + new - decommissioned. The model may decommission the existing PV (it can no longer produce and only costs fixed O&M), so installed PV can be zero even though PV existed before.

In [ ]:
case = "ext_no_pv"
if ready(case):
    pv = {}
    for name in [REFERENCE, case]:
        g = symbol(name, "G_CAP_YCRAF")
        by_category = g[(g["Y"] == YEAR) & (g["TECH_TYPE"] == "SOLAR-PV")].groupby("VARIABLE_CATEGORY")["Val"].sum()
        pv[name] = {"existing PV (GW)": by_category.get("EXOGENOUS", 0.0),
                    "decommissioned PV (GW)": by_category.get("DECOMMISSIONING", 0.0),
                    "new PV capacity (GW)": capacity(name, "new", tech="SOLAR-PV").sum(),
                    "installed PV capacity (GW)": capacity(name, "installed", tech="SOLAR-PV").sum()}
        pro = symbol(name, "PRO_YCRAGF")
        pv[name]["PV generation (TWh)"] = pro[(pro["Y"] == YEAR) & (pro["TECH_TYPE"] == "SOLAR-PV")]["Val"].sum()
    table = pd.DataFrame(pv).rename(columns={REFERENCE: "reference", case: CASES[case]})
    table["change"] = table[CASES[case]] - table["reference"]
    display(table.style.format("{:,.1f}"))

    # new and installed PV capacity by country
    for kind, label in [("new", "New PV capacity"), ("installed", "Installed PV capacity")]:
        compare_bars(case, capacity(REFERENCE, kind, tech="SOLAR-PV", by="C"), capacity(case, kind, tech="SOLAR-PV", by="C"),
                     f"{label} {YEAR} by country (countries above 0.5 GW)", min_value=0.5)

In [ ]:
if ready("ext_no_pv"):
    # what replaces the PV: installed capacity by technology
    installed_capacity_figure("ext_no_pv")
    plot_objective("ext_no_pv")

### Transmission map without PV

Directed transmission lines of the simulated year: existing plus new capacity, width proportional to capacity. Without PV the system may build more or different transmission. The right-hand map shows the capacity change per directed line (green = more, red = less).

In [ ]:
case = "ext_no_pv"
if ready(case):
    ref_links, case_links = links(REFERENCE), links(case)
    change = case_links.sub(ref_links, fill_value=0.0)
    same = lambda a, b, v: LINE_COLOR
    show_maps([("Reference", ref_links, same), (CASES[case], case_links, same),
               ("Change in capacity per directed line (GW)", change, lambda a, b, v: MORE_COLOR if v > 0 else FRANCE_COLOR)])
    total = pd.DataFrame({"reference": [ref_links.sum()], CASES[case]: [case_links.sum()]}, index=["transmission capacity (GW, both directions counted)"])
    total["change"] = total[CASES[case]] - total["reference"]
    display(total.style.format("{:,.1f}"))
    biggest = change[change.abs() >= 0.05].sort_values(key=abs, ascending=False).head(10)
    biggest.index = [f"{a} -> {b}" for a, b in biggest.index]
    print("Ten largest capacity changes per directed line (GW)")
    display(biggest.to_frame("change").style.format("{:+,.2f}"))

### CO2 emissions without PV

`EMI_YCRAG` holds the annual CO2 emissions of the simulated year (kton) per region, area, technology and fuel. Without PV, more fuel-based generation is expected, so emissions should rise.

In [ ]:
def emissions(scenario, by=None):
    "CO2 emissions (Mt) of the simulated year, in total or grouped by `by` (FFF = fuel, C = country, TECH_TYPE)."
    e = symbol(scenario, "EMI_YCRAG")
    e = e[e["Y"] == YEAR]
    return e["Val"].sum() / 1000 if by is None else e.groupby(by)["Val"].sum() / 1000


case = "ext_no_pv"
if ready(case):
    total = pd.DataFrame({"reference": [emissions(REFERENCE)], CASES[case]: [emissions(case)]}, index=["CO2 emissions (Mt)"])
    total["change"] = total[CASES[case]] - total["reference"]
    total["change %"] = total["change"] / total["reference"] * 100
    display(total.style.format({"reference": "{:,.1f}", CASES[case]: "{:,.1f}", "change": "{:+,.1f}", "change %": "{:+.1f}"}))

    fuel = compare_bars(case, emissions(REFERENCE, "FFF"), emissions(case, "FFF"), f"CO2 emissions {YEAR} by fuel", xlabel="Mt CO2",
                        min_value=0.05, panel_widths=(1.6, 1.3))
    country = compare_bars(case, emissions(REFERENCE, "C"), emissions(case, "C"), f"CO2 emissions {YEAR} by country (above 1 Mt)", xlabel="Mt CO2",
                           min_value=1.0)

## 4. Onshore wind investment cost x 10 (`ext_onshore_wind_capex_x10`)

Expected: far less new onshore wind capacity (replaced by other technologies), existing onshore wind unchanged, higher objective.

In [ ]:
case = "ext_onshore_wind_capex_x10"
if ready(case):
    installed_capacity_figure(case)
    table = pd.DataFrame({
        "reference": [capacity(REFERENCE, "new", tech="WIND-ON").sum(), capacity(REFERENCE, "installed", tech="WIND-ON").sum()],
        CASES[case]: [capacity(case, "new", tech="WIND-ON").sum(), capacity(case, "installed", tech="WIND-ON").sum()]},
        index=["new onshore wind (GW)", "installed onshore wind (GW)"])
    table["change"] = table[CASES[case]] - table["reference"]
    display(table.style.format("{:,.1f}"))
    plot_objective(case)

## 5. No transmission expansion (`ext_no_transmission_expansion`)

Expected: no endogenous (new) transmission capacity, more generation capacity close to demand, higher objective.

In [ ]:
case = "ext_no_transmission_expansion"
if ready(case):
    installed_capacity_figure(case)
    rows = {}
    for name in [REFERENCE, case]:
        x = symbol(name, "X_CAP_YCR")
        x = x[x["Y"] == YEAR]
        rows[name] = x.groupby("VARIABLE_CATEGORY")["Val"].sum()
    table = pd.DataFrame(rows).rename(columns={REFERENCE: "reference", case: CASES[case]}).fillna(0.0)
    table.loc["total"] = table.sum()
    table["change"] = table[CASES[case]] - table["reference"]
    print("Electric transmission capacity (GW, both directions counted) by category")
    display(table.style.format("{:,.1f}"))
    plot_objective(case)

## 6. No France interconnectors (`ext_no_france_interconnectors`)

Expected: no capacity and no exchange between France and its neighbours, higher objective. The maps show the directed transmission lines of the simulated year (existing plus new capacity): one arrow per direction, width proportional to capacity, France's links in red. Region positions are approximate region centres. The right-hand map shows the capacity change per directed line (green = more, red = less).

In [ ]:
case = "ext_no_france_interconnectors"
if ready(case):
    ref_links, case_links = links(REFERENCE), links(case)
    change = case_links.sub(ref_links, fill_value=0.0)
    show_maps([("Reference", ref_links, None),
               (CASES[case], case_links, None),
               ("Change in capacity per directed line (GW)", change, lambda a, b, v: MORE_COLOR if v > 0 else FRANCE_COLOR)])

    fr = pd.DataFrame({"reference": ref_links, CASES[case]: case_links}).fillna(0.0)
    fr = fr[[("FR" in idx) for idx in fr.index]]
    fr = fr[fr.abs().max(axis=1) > 0]
    if fr[CASES[case]].sum() > 0:
        print(f"WARNING: {fr[CASES[case]].sum():.1f} GW of links of France are still present in the scenario results. "
              "Check that the run used the current data files of ext_no_france_interconnectors.")
    fr["change"] = fr[CASES[case]] - fr["reference"]
    fr.index = [f"{a} -> {b}" for a, b in fr.index]
    print("Links of France (GW)")
    display(fr.style.format("{:,.2f}"))
    print(f"Capacity on directed lines other than France's links: reference {ref_links[[('FR' not in i) for i in ref_links.index]].sum():,.1f} GW, "
          f"scenario {case_links[[('FR' not in i) for i in case_links.index]].sum():,.1f} GW")
    plot_objective(case)